# RQ2b results figures -- plan

Builds the figure backing RQ2b's attribution claims in
`documentation/august_draft/5_Chapter_results_evaluation/f_written_draft_v1_18thaug.tex`.
4survey only, by design -- RQ2b has no 6survey arm anywhere in this project.

**Figure in this notebook**

| Figure | Status | Question it answers |
|---|---|---|
| R2b-1 | **Must-have** | Which variables are large AND stable across model families, and does adding more environmental data close the leftover spatial gap? |
| R2b-4 (new) | Added 2026-08-20 | What does a plot's own trajectory actually look like when it persistently sits above/below the shared curve? |
| R2b-5 (new) | Added 2026-08-20 | Does EN/XGBoost systematically over- or under-predict a plot's departure from the shared curve? |
| R2b-6 (new) | Added 2026-08-20 | Two equally-confounded-by-eye variables (topex, chelsa_bio12_precip_mm) -- sets up the mechanism question. |
| R2b-7 (new) | Added 2026-08-20 | Does each variable's signal survive removing compartment structure? -- resolves the mechanism. |
| R2b-8 (new, appendix) | Added 2026-08-20 | CanopyCover as the low-confounding calibration contrast to R2b-6. |

**Not built here (cut for time)**: R2b-2 (CanopyCover-dropped ablation) and R2b-3 (VIF diagnostic)
were both considered and dropped -- both add nothing beyond what is already precisely stated in
prose (e.g. EN R2 0.350->0.231 on CanopyCover removal), so building either would spend real time
on a confirmation-only appendix figure. If you want them later, the numbers already live in
`TEMP_results/TEMP_rq2_attribution_results_2026-08-11.tex`.

**Style / uncertainty convention**: see `notebooks/results_figures_style.py`. Panel A's whiskers
are fold-to-fold **sample SD** (5 folds), not a CI. Panel B (Moran's I) has no CI computed
anywhere in this project -- shown as a point value with p annotated as text. Panel C shows one
pooled value per compartment (across the 5 test folds), no per-compartment CI.

In [ ]:
# Purpose: make the models/ package (and notebooks/results_figures_style.py) importable from
# this notebook. Same convention already used across this project's other notebooks (e.g.
# notebooks/model_results/baseline_results.ipynb): walk upward until a folder containing both
# README.md and data/ is found -- that is the project root.

import sys
from pathlib import Path

notebook_directory = Path.cwd().resolve()
project_root = next(
    folder for folder in [notebook_directory, *notebook_directory.parents]
    if (folder / "README.md").exists() and (folder / "data").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("project_root:", project_root)

In [ ]:
import json

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from models.common.geo import load_compartment_boundaries, load_plot_coordinates
from models.growth_curve_attribution.residual_spatial_autocorrelation_check import compute_residual_morans_i
from notebooks.results_figures_style import COLOR_EN, COLOR_XGBOOST, COLOR_NEUTRAL_EDGE, DIVERGING_CMAP, apply_rcparams

apply_rcparams()

FIGURES_DIR = project_root / "figures" / "fig_results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
COHORT = "4survey"  # RQ2b has no 6survey arm
SET_RUN_NAMES = {
    "Set1": "rq2_attribution_nested_set1_baseline",
    "Set2": "rq2_attribution_nested_set2_top10",
    "Set3": "rq2_attribution_nested_set3_gated_terrain_wind_vif",
    "Set4": "rq2_attribution_nested_set4_gated_all_vif",
}

## Figure R2b-1 -- what global attribution finds, and what it still can not explain

**Research question**: RQ2b items 1-3 -- CanopyCover/thinning dominate across three converging
methods; `slope_degrees` is stable across NLME/EN while `topex` is stable only within NLME; does
the residual's own spatial clustering (Moran's I) shrink as more environmental data is added?

**Data**: Panel A -- Set4 `elastic_net_coefficients.csv` and `nlme_fixed_effects.json`, all 5
folds, `outputs/spatial_block_kfold/rq2_attribution_nested_set4_gated_all_vif/4survey/fold_{0-4}/`.
Panel B -- pooled 5-fold `predictions.csv` (`elastic_net_predicted`, `xgboost_predicted` columns)
for all four sets, joined to plot coordinates, residual Moran's I computed live via the project's
own `compute_residual_morans_i()` -- same function/weights used for every other Moran's I check in
this project. Panel C -- Set4's own pooled EN residual, aggregated to compartment mean, mapped via
`load_compartment_boundaries()`.

**Encoding**: 3-panel composite. Panel A: coefficient forest plot, curated to the 8 variables
actually discussed in the chapter's own prose (CanopyCover, the 2 thinning columns,
slope_degrees, topex, tas_mean, soilgrids_ph, chelsa_bio12_precip_mm -- not the full ~19-variable
Set4 set), EN and LMM as two marker shapes, Set3 (hollow) and Set4 (filled) as two sub-columns per
row -- the visual payoff of the slope-stable/topex-unstable contrast needs both sets shown, since
the claim is that it HOLDS ACROSS sets, not a Set4-only snapshot. Panel B: Moran's I by set, one
line per method (EN, XGBoost). Panel C: diverging compartment-mean EN residual map, centred at 0
(EN chosen deliberately, not XGBoost -- see this section's own note on why).

**Caption must state**: Panel C shows leftover unexplained error, not a spatially-varying
relationship -- unlike RQ3's GNNWR, none of RQ2b's methods (LMM, EN, XGBoost) produce a
coefficient that varies by location.

In [ ]:
# --- Panel A data: EN + LMM coefficients, Set3 AND Set4, mean +/- SD across 5 folds -- curated
# to the 8 variables actually discussed in prose, not the full Set4 variable list. ---
CURATED_VARIABLES = [
    "CanopyCover", "time_since_thinning", "time_since_thinning_missing",
    "slope_degrees", "topex", "tas_mean", "soilgrids_ph", "chelsa_bio12_precip_mm",
]

def load_set_coefficients(set_label):
    set_dir = project_root / "outputs" / "spatial_block_kfold" / SET_RUN_NAMES[set_label] / COHORT
    en_frames, nlme_frames = [], []
    for fold in range(N_FOLDS):
        fold_dir = set_dir / f"fold_{fold}"
        en_coef = pd.read_csv(fold_dir / "elastic_net_coefficients.csv", index_col=0)
        en_coef.columns = ["coefficient"]
        en_coef["fold"] = fold
        en_frames.append(en_coef.reset_index(names="variable"))

        with open(fold_dir / "nlme_fixed_effects.json") as f:
            nlme_json = json.load(f)["fixed_effects"]
        nlme_frames.append(pd.DataFrame(
            [{"variable": var, "coefficient": vals["coefficient"], "fold": fold} for var, vals in nlme_json.items()]
        ))
    en_all = pd.concat(en_frames, ignore_index=True)
    nlme_all = pd.concat(nlme_frames, ignore_index=True)
    en_summary = en_all.groupby("variable")["coefficient"].agg(["mean", "std"]).rename(
        columns={"mean": "en_mean", "std": "en_sd"})
    nlme_summary = nlme_all.groupby("variable")["coefficient"].agg(["mean", "std"]).rename(
        columns={"mean": "nlme_mean", "std": "nlme_sd"})
    # NOT .fillna(0) -- Set3 is terrain/wind only and genuinely has no climate/soil columns
    # (tas_mean/soilgrids_ph/chelsa_bio12_precip_mm only exist in Set4). Filling with 0 would
    # plot a fake "Set3 coefficient is exactly zero" point instead of correctly showing those
    # variables as structurally absent from Set3. Left as NaN; the plotting loop skips them.
    combined = en_summary.join(nlme_summary, how="outer")
    return combined.reindex(CURATED_VARIABLES)

panel_a_set3 = load_set_coefficients("Set3")
panel_a_set4 = load_set_coefficients("Set4")
print("Set3:\n", panel_a_set3)
print("Set4:\n", panel_a_set4)

In [ ]:
# --- Panel B + C data: pooled predictions per set, residual, Moran's I, and Set4's own map ---
coordinates = load_plot_coordinates()

panel_b_rows = []
set4_pooled_with_xy = None
for set_label, run_name in SET_RUN_NAMES.items():
    frames = []
    for fold in range(N_FOLDS):
        p = project_root / "outputs" / "spatial_block_kfold" / run_name / COHORT / f"fold_{fold}" / "predictions.csv"
        frames.append(pd.read_csv(p))
    pooled = pd.concat(frames, ignore_index=True)
    pooled = pooled[pooled["split"] == "test"] if "split" in pooled.columns else pooled
    pooled = pooled.merge(coordinates, on="identification", how="left")
    pooled["en_residual"] = pooled["observed"] - pooled["elastic_net_predicted"]
    pooled["xgb_residual"] = pooled["observed"] - pooled["xgboost_predicted"]

    # compute_residual_morans_i returns 5 values (semivariogram-informed distance-band
    # convention) -- range_m differs per residual field, so it is kept alongside each row rather
    # than assumed to match across methods/sets.
    en_i, en_p, en_n, en_range_m, en_status = compute_residual_morans_i(pooled["x"], pooled["y"], pooled["en_residual"])
    xgb_i, xgb_p, xgb_n, xgb_range_m, xgb_status = compute_residual_morans_i(pooled["x"], pooled["y"], pooled["xgb_residual"])
    panel_b_rows.append({"set": set_label, "method": "Elastic Net", "morans_i": en_i, "p_value": en_p, "range_m": en_range_m})
    panel_b_rows.append({"set": set_label, "method": "XGBoost", "morans_i": xgb_i, "p_value": xgb_p, "range_m": xgb_range_m})
    print(f"{set_label}: EN Moran's I={en_i:.3f} (p={en_p:.3f}, range={en_range_m:.0f}m), "
          f"XGB Moran's I={xgb_i:.3f} (p={xgb_p:.3f}, range={xgb_range_m:.0f}m)")

    if set_label == "Set4":
        set4_pooled_with_xy = pooled  # reused directly for Panel C, no re-pooling

panel_b_data = pd.DataFrame(panel_b_rows)

In [ ]:
fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(2, 2, width_ratios=[1.1, 1], height_ratios=[1.6, 1])
ax_a = fig.add_subplot(gs[:, 0])
ax_b = fig.add_subplot(gs[1, 1])
ax_c = fig.add_subplot(gs[0, 1])

# --- Panel A: coefficient forest plot, Set3 (hollow) vs. Set4 (filled), EN (circle) vs. LMM
# (square) -- 4 points per variable row, showing the slope-stable/topex-unstable contrast holds
# across sets, not just at Set4. ---
y_positions = np.arange(len(CURATED_VARIABLES))
row_offsets = {"Set3": -0.15, "Set4": 0.15}
fill_style = {"Set3": "none", "Set4": "full"}
for set_label, panel_a_data in [("Set3", panel_a_set3), ("Set4", panel_a_set4)]:
    offset = row_offsets[set_label]
    ax_a.errorbar(panel_a_data["en_mean"], y_positions + offset - 0.04, xerr=panel_a_data["en_sd"],
                  fmt="o", color=COLOR_EN, markersize=5, capsize=2, fillstyle=fill_style[set_label],
                  label=f"Elastic Net ({set_label})")
    ax_a.errorbar(panel_a_data["nlme_mean"], y_positions + offset + 0.04, xerr=panel_a_data["nlme_sd"],
                  fmt="s", color=COLOR_NEUTRAL_EDGE, markersize=5, capsize=2, fillstyle=fill_style[set_label],
                  label=f"LMM ({set_label})")
panel_a_data = panel_a_set4  # y-tick labels only need one set's index (both share CURATED_VARIABLES)
ax_a.axvline(0, color="black", linewidth=0.8)
ax_a.set_yticks(y_positions)
ax_a.set_yticklabels(panel_a_data.index, fontsize=10)
ax_a.set_xlabel("Standardised coefficient (mean +/- SD, 5 folds; hollow=Set3, filled=Set4)")
ax_a.legend(fontsize=6.5, frameon=False, loc="lower left", ncol=2)

# --- Panel B: Moran's I by set ---
for method, color in [("Elastic Net", COLOR_EN), ("XGBoost", COLOR_XGBOOST)]:
    subset = panel_b_data[panel_b_data["method"] == method]
    ax_b.plot(subset["set"], subset["morans_i"], marker="o", color=color, label=method, linewidth=1.8)
ax_b.set_ylim(0, 0.2)  # corrected semivariogram Moran's I range is 0.03-0.15, not the old k=8 0-0.8 range
ax_b.set_ylabel("Residual Moran's I")
# XGBoost's own line is NOT monotonic (dips at Set2, rises at Set3, drops again at Set4) -- the
# 63% figure is a Set1-vs-Set4 endpoint comparison, not a smooth trend; say so directly rather
# than let the line visually imply a cleaner decline than what's actually there.
ax_b.set_title("Panel B: spatial clustering by set\n(XGBoost: net Set1->Set4 change, not monotonic)", fontsize=9.5)
ax_b.legend(fontsize=8, frameon=False)

# --- Panel C: Set4 EN residual, compartment mean, mapped ---
compartment_mean = set4_pooled_with_xy.groupby("cpmt")["en_residual"].mean().reset_index()
boundaries = load_compartment_boundaries()
mapped = boundaries.merge(compartment_mean, on="cpmt", how="left")
# Robust colour range (98th percentile of |value|, not the true max) -- the true max is likely
# 1-2 outlier compartments stretching the scale and washing out contrast for the typical range,
# same fix already used on R3-2's deviation map.
vmax = mapped["en_residual"].abs().quantile(0.98)
norm = mpl.colors.TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)
mapped.plot(column="en_residual", cmap=DIVERGING_CMAP, norm=norm, ax=ax_c,
            edgecolor="white", linewidth=0.2, missing_kwds={"color": "#DDDDDD"})
sm = plt.cm.ScalarMappable(cmap=DIVERGING_CMAP, norm=norm)
fig.colorbar(sm, ax=ax_c, shrink=0.8, label="Mean EN residual (m)\nleftover error, not a local effect")
ax_c.set_title("Panel C: Set4 leftover residual (compartment mean)", fontsize=10)
ax_c.set_aspect("equal")
ax_c.set_xticks([]); ax_c.set_yticks([])
# Tighten the map extent to the actual data bounding box (small margin) instead of matplotlib's
# default padding -- the previous export had a lot of dead whitespace on both sides.
data_bounds = boundaries[boundaries["cpmt"].isin(compartment_mean["cpmt"])].total_bounds
x_margin = (data_bounds[2] - data_bounds[0]) * 0.03
y_margin = (data_bounds[3] - data_bounds[1]) * 0.03
ax_c.set_xlim(data_bounds[0] - x_margin, data_bounds[2] + x_margin)
ax_c.set_ylim(data_bounds[1] - y_margin, data_bounds[3] + y_margin)

fig.suptitle("Global attribution: what it finds (A), and what it still cannot explain (B, C)", fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "q1_attribution_composite.png", dpi=200)
plt.show()

## Figure R2b-4 (new) -- example plots by residual tier

**Research question**: RQ2b asks "what explains a plot's persistent departure from the shared
curve" -- every existing RQ2b figure answers this only in aggregate (coefficients, R2, Moran's I).
This is the first figure that shows what "departure from the shared curve" actually looks like for
a real plot's own observed trajectory.

**Data**: `models.xgb_environmental.data.load_plots_for_cohort("4survey")` for the real per-plot
`mean_cr_residual` target (71,766 plots, already verified against the known population count) --
sort by this value, take the 2 most positive (persistently ABOVE the shared curve), 2 most negative
(persistently BELOW), 2 nearest zero (typical/well-fit). Raw per-survey rows via
`load_filtered_growth_curve_table("4survey")`. Benchmark curve: the pooled 4survey Chapman-Richards
fit (y_max=51.963, k=0.010371, p=0.865 -- the same frozen anchor cited throughout this chapter).

**Encoding**: 6 small multiples (2 per tier), each plot's own observed trajectory (solid) against
the single POOLED curve (dashed) -- deliberately the pooled curve, not a yield-class benchmark
(that is RQ3's target, not RQ2b's), so this figure visualizes exactly what `mean_cr_residual` means.

In [ ]:
from models.xgb_environmental.data import load_plots_for_cohort
from models.growth_curve_attribution.data import load_growth_curve_table
from models.chapman_richards.chapman_richards import chapman_richards

CR_PARAMS_4SURVEY = {"y_max": 51.963, "k": 0.010371, "p": 0.865}  # pooled anchor, cited throughout this chapter

plots_with_residual = load_plots_for_cohort("4survey")
above = plots_with_residual.sort_values("mean_cr_residual", ascending=False).head(2)
below = plots_with_residual.sort_values("mean_cr_residual", ascending=True).head(2)
typical = plots_with_residual.iloc[(plots_with_residual["mean_cr_residual"]).abs().argsort()[:2]]

tier_plots = list(above["identification"]) + list(typical["identification"]) + list(below["identification"])
tier_labels = ["above curve", "above curve", "typical", "typical", "below curve", "below curve"]
tier_residuals = list(above["mean_cr_residual"]) + list(typical["mean_cr_residual"]) + list(below["mean_cr_residual"])
print("Selected plots by tier:", list(zip(tier_plots, tier_labels, [round(r, 2) for r in tier_residuals])))

# NOT load_filtered_growth_curve_table() -- that applies RQ3\'s maturity gate (Age>=30 at final
# survey), a DIFFERENT population than mean_cr_residual\'s own 71,766-plot population. Using the
# filtered table silently dropped 3 of 6 selected plots (confirmed: 0 rows in filtered, 4 rows in
# raw, for exactly the plots that showed no observed line). Raw table matches the target\'s own
# population instead.
growth_rows_rq2b = load_growth_curve_table("4survey")

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(18, 3.2), sharey=False)
age_range = np.linspace(0, 100, 200)
cr_curve = chapman_richards(age_range, CR_PARAMS_4SURVEY["y_max"], CR_PARAMS_4SURVEY["k"], CR_PARAMS_4SURVEY["p"])

for ax, plot_id, label, residual in zip(axes, tier_plots, tier_labels, tier_residuals):
    plot_rows = growth_rows_rq2b[growth_rows_rq2b["identification"] == plot_id].sort_values("Age")
    ax.plot(age_range, cr_curve, linestyle="--", color=COLOR_NEUTRAL_EDGE, linewidth=1.2, label="pooled CR curve")
    ax.plot(plot_rows["Age"], plot_rows["elev_percentile_95th"], marker="o", color=COLOR_XGBOOST,
            linewidth=1.6, label="observed")
    ax.set_title(f"{label}\nplot {int(plot_id)}, mean resid={residual:+.1f}m", fontsize=8)
    ax.set_xlabel("Age (years)", fontsize=8)
    ax.set_xlim(0, 100)

axes[0].set_ylabel("Top height (m)")
axes[0].legend(fontsize=7, frameon=False)
fig.suptitle("Example plots by residual tier -- what persistent departure from the pooled curve looks like", fontsize=11)
plt.tight_layout()
plt.show()

## Figure R2b-5 (new) -- does the model over- or under-predict?

**Research question**: complements RQ2b's coefficient-direction story with an actual bias check --
none of the existing RQ2b figures show whether EN/XGBoost systematically over- or under-predict
`mean_cr_residual`, only which variables matter and by how much.

**Data**: reuses the same pooled Set4 predictions already loaded for Figure R2b-1's Panel B/C
(`set4_pooled_with_xy`, columns `observed`, `elastic_net_predicted`, `xgboost_predicted`) -- no new
data, no new fitting.

**Encoding**: predicted vs. observed scatter, one panel per method, 1:1 reference line (perfect
prediction). Points above the line = under-prediction (model predicts less departure than real);
below = over-prediction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
axis_limits = (set4_pooled_with_xy["observed"].min() - 1, set4_pooled_with_xy["observed"].max() + 1)

for ax, pred_col, label, color in [
    (axes[0], "elastic_net_predicted", "Elastic Net", COLOR_EN),
    (axes[1], "xgboost_predicted", "XGBoost", COLOR_XGBOOST),
]:
    ax.scatter(set4_pooled_with_xy["observed"], set4_pooled_with_xy[pred_col],
               s=3, alpha=0.15, color=color, edgecolor="none")
    ax.plot(axis_limits, axis_limits, color=COLOR_NEUTRAL_EDGE, linewidth=1.2, linestyle="--",
            label="perfect prediction (1:1)")
    ax.set_xlim(axis_limits); ax.set_ylim(axis_limits)
    ax.set_aspect("equal")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Observed mean_cr_residual (m)")

axes[0].set_ylabel("Predicted mean_cr_residual (m)")
axes[0].legend(fontsize=8, frameon=False)
fig.suptitle("Does the model over- or under-predict? (points above line = under-prediction)", fontsize=11)
plt.tight_layout()
plt.show()

## Figure R2b-6 (new) -- two variables, equally spatially confounded by eye

**Research question**: sets up the mechanism question the next figure resolves -- topex and
chelsa_bio12_precip_mm have similar high ICC (0.74 / 0.97) and look equally "blocky" by eye, so
why does only one destabilise the Elastic Net coefficient?

**Data**: `load_plots_for_cohort("4survey")` for raw per-plot `topex`/`chelsa_bio12_precip_mm`
values + x/y/cpmt, `load_compartment_boundaries()` for the full-extent boundary overlay (all 231
compartments -- this figure's whole point is showing blockiness generally, unlike R3-2's
restricted-to-flagged-compartments treatment).

**Encoding**: two-panel map, same colour treatment on both for direct visual comparison, full
compartment boundary lines (thin, light) on both.

In [ ]:
plots_for_maps = load_plots_for_cohort("4survey")
all_boundaries = load_compartment_boundaries()

fig, (ax_m1, ax_m2) = plt.subplots(1, 2, figsize=(13, 6))
for ax, column, title in [
    (ax_m1, "topex", "topex (ICC=0.74)"),
    (ax_m2, "chelsa_bio12_precip_mm", "chelsa_bio12_precip_mm (ICC=0.97)"),
]:
    sc = ax.scatter(plots_for_maps["x"], plots_for_maps["y"], c=plots_for_maps[column],
                     cmap="viridis", s=4, alpha=0.7)
    all_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.3, alpha=0.6, zorder=3)
    fig.colorbar(sc, ax=ax, shrink=0.75, label=column)
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Two variables, equally spatially confounded by eye -- both align with compartment boundaries", fontsize=11)
plt.tight_layout()
plt.show()

## Figure R2b-7 (new) -- does each variable's signal survive removing compartment structure?

**Research question**: resolves the mechanism the previous figure sets up -- does real,
generalisable local signal exist for each variable once between-compartment differences are
removed, or does the raw bivariate correlation evaporate?

**Data**: live-computed, same method and source as
`TEMP_results_attribution/TEMP_rq2b_topex_tasmean_nonlinearity_check_2026-08-20.tex` (verified to
reproduce its exact cited numbers before building this figure: topex 0.093->0.210, tas_mean
0.273->0.082, soilgrids_ph 0.271->0.046, chelsa_bio12_precip_mm -0.364->-0.114). Within-compartment
= subtract each compartment's own mean from BOTH the variable and `mean_cr_residual_4survey`
before computing Spearman correlation -- mechanically close to what LMM's random intercept absorbs.

**Encoding**: dumbbell/slope chart, y-axis = signed Spearman correlation, zero line marked, one
row per variable, two points (raw, within-compartment) connected by a line.

In [ ]:
from scipy import stats

env_df = pd.read_parquet(project_root / "data" / "processed" / "environmental" / "plot_environmental_features.parquet")

MECHANISM_VARIABLES = ["topex", "tas_mean_4survey", "soilgrids_ph", "chelsa_bio12_precip_mm"]
mechanism_results = {}
for col in MECHANISM_VARIABLES:
    sub = env_df.dropna(subset=["mean_cr_residual_4survey", col, "cpmt"]).copy()
    raw_rho = stats.spearmanr(sub[col], sub["mean_cr_residual_4survey"])[0]

    within_col = sub[col] - sub.groupby("cpmt")[col].transform("mean")
    within_target = sub["mean_cr_residual_4survey"] - sub.groupby("cpmt")["mean_cr_residual_4survey"].transform("mean")
    within_rho = stats.spearmanr(within_col, within_target)[0]

    mechanism_results[col] = {"raw": raw_rho, "within_compartment": within_rho}
    print(f"{col}: raw={raw_rho:+.3f}, within-compartment={within_rho:+.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
variable_labels = ["topex", "tas_mean", "soilgrids_ph", "chelsa_bio12_precip_mm"]
stability = {"topex": "unstable", "tas_mean_4survey": "unstable", "soilgrids_ph": "unstable",
             "chelsa_bio12_precip_mm": "stable"}
color_by_stability = {"unstable": COLOR_XGBOOST, "stable": COLOR_EN}

for i, (col, label) in enumerate(zip(MECHANISM_VARIABLES, variable_labels)):
    raw = mechanism_results[col]["raw"]
    within = mechanism_results[col]["within_compartment"]
    color = color_by_stability[stability[col]]
    ax.plot([raw, within], [i, i], color=color, linewidth=1.5, zorder=1)
    ax.scatter([raw], [i], marker="o", color=color, s=60, zorder=2, label="raw" if i == 0 else None)
    ax.scatter([within], [i], marker="D", color=color, s=60, zorder=2, edgecolor="black", linewidth=0.5,
               label="within-compartment" if i == 0 else None)

ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(range(len(variable_labels)))
ax.set_yticklabels([f"{lbl} ({stability[col]})" for lbl, col in zip(variable_labels, MECHANISM_VARIABLES)])
ax.set_xlabel("Spearman correlation with mean_cr_residual_4survey")
ax.legend(fontsize=8, frameon=False, loc="upper left")
fig.suptitle("Does real local signal survive removing compartment structure?\n"
             "(circle=raw, diamond=within-compartment; EN-stable variable in EN colour, unstable in XGBoost colour)",
             fontsize=10)
plt.tight_layout()
plt.show()

## Figure R2b-8 (new, appendix) -- a low-confounding contrast

**Research question**: calibration reference for Figure R2b-6 -- what does genuinely LOW spatial
confounding look like, next to what topex/chelsa_bio12_precip_mm (both high-ICC) look like?

**Data**: `CanopyCover` from the same `plots_for_maps` table already loaded above. ICC=0.30,
70,769 distinct values (both verified live, matching the corrected draft text -- see this
session's TODO note on the topex distinct-value correction).

**Encoding**: single map, same colour/boundary treatment as R2b-6, for direct visual contrast.
Should read as visibly speckled within each compartment, not blocky.

In [ ]:
fig, ax_canopy = plt.subplots(figsize=(7, 6))
sc = ax_canopy.scatter(plots_for_maps["x"], plots_for_maps["y"], c=plots_for_maps["CanopyCover"],
                        cmap="viridis", s=4, alpha=0.7)
all_boundaries.boundary.plot(ax=ax_canopy, color="black", linewidth=0.3, alpha=0.6, zorder=3)
fig.colorbar(sc, ax=ax_canopy, shrink=0.8, label="CanopyCover")
ax_canopy.set_title("CanopyCover (ICC=0.30, ~70,769 distinct values) -- a low-confounding contrast", fontsize=10)
ax_canopy.set_aspect("equal")
ax_canopy.set_xticks([]); ax_canopy.set_yticks([])
plt.tight_layout()
plt.show()